# Generators – Practice Exercises

This notebook contains hands-on exercises to practice **generators and lazy iteration** in Python,
following the concepts from `g_generators.ipynb`.

For each exercise:

- Read the description carefully.
- Implement your solution in the provided code cell.
- Add tests or a small demo at the bottom of the cell to verify your solution.

### Contents

- [Exercise 1 – Simple Generator Function](#exercise-1--simple-generator-function)
- [Exercise 2 – Range-Like Generator](#exercise-2--range-like-generator)
- [Exercise 3 – Filter Generator](#exercise-3--filter-generator)
- [Exercise 4 – Generator Expression Pipeline](#exercise-4--generator-expression-pipeline)
- [Exercise 5 – Read Lines from a File-Like Object](#exercise-5--read-lines-from-a-file-like-object)
- [Exercise 6 – Parse CSV-Like Lines into Objects](#exercise-6--parse-csv-like-lines-into-objects)
- [Exercise 7 – Chained Generators (Filter + Map)](#exercise-7--chained-generators-filter--map)
- [Exercise 8 – Sliding Window Generator](#exercise-8--sliding-window-generator)
- [Exercise 9 – Flatten Multiple Iterables with `yield from`](#exercise-9--flatten-multiple-iterables-with-yield-from)

## Exercise 1 – Simple Generator Function

**Goal**: Write a generator function that yields the first **n** square numbers (e.g. 1, 4, 9, 16, …).

Requirements:

- Function signature: `squares(n: int)`.
- Yield each square one at a time; do not build a list of all squares in memory.
- Demonstrate that it works by iterating over the generator (e.g. with a `for` loop or `list(squares(5))`).

Hint: Use a loop and `yield` the next square inside it.

In [1]:
# Exercise 1 – implement squares(n) here

from collections.abc import Iterator

def squares(n: int) -> Iterator[int]:
    for i in range(n):
        yield (i + 1) ** 2
    

# Demo:
for x in squares(5):
    print(x)
print(list(squares(3)))

1
4
9
16
25
[1, 4, 9]


## Exercise 2 – Range-Like Generator

**Goal**: Implement a generator that yields a sequence of numbers from **start** by repeatedly adding **step**, for **count** values (similar in spirit to `range`, but with a fixed count and float-friendly step).

Requirements:

- Signature: `seq(start: float, step: float, count: int)`.
- First value yielded is `start`, then `start + step`, then `start + 2*step`, … up to `count` values.
- Use only a loop and `yield`; no list accumulation.

Hint: `for i in range(count): yield start + i * step`.

In [7]:
# Exercise 2 – implement seq(start, step, count) here

from collections.abc import Iterator

def seq(start: float, step: float, count: int) -> Iterator[float]:
    for i in range(count): yield start + i * step

# Demo: list(seq(10.0, 0.5, 4))  ->  [10.0, 10.5, 11.0, 11.5]
args = (10.0, 0.5, 4)
print("Original exercise: with fixed step...")
print(f"seq{args} = {list(seq(*args))}")

def seq(start: float, stop: float, count: int) -> Iterator[float]:
    step = (stop - start) / (count - 1)
    for i in range(count): yield start + i * step
    
args = (10.0, 11.5, 4)
print("Modified exercise: with fixed limits...")
print(f"seq{args} = {list(seq(*args))}")

Original exercise: with fixed step...
seq(10.0, 0.5, 4) = [10.0, 10.5, 11.0, 11.5]
Modified exercise: with fixed limits...
seq(10.0, 11.5, 4) = [10.0, 10.5, 11.0, 11.5]


## Exercise 3 – Filter Generator

**Goal**: Write a generator function that takes an **iterable** and a **predicate** (a callable that returns True/False), and yields only the items for which the predicate is True.

Requirements:

- Signature: `filter_gen(iterable, predicate)`.
- Do not use the built-in `filter`; implement the logic with a loop and `yield`.
- Test with a simple iterable (e.g. numbers) and a predicate (e.g. "is even", "greater than 5").

Hint: Iterate over the iterable and `yield item` only when `predicate(item)` is true.

In [20]:
# Exercise 3 – implement filter_gen(iterable, predicate) here

from collections.abc import Iterator
from typing import Callable

def filter_gen(iterable: Iterator, predicate: Callable) -> Iterator[int]:
    for element in iterable:
        if not predicate(element): continue
        yield element

# Demo: list(filter_gen([1,2,3,4,5], lambda x: x % 2 == 0))  ->  [2, 4]
iterable = [1,2,3,4,5]
predicate = lambda x: (x % 2 == 0)
args = (iterable, predicate)
result = list(filter_gen(*args))
print(f"filter_gen{args} = {result}")

filter_gen([1, 2, 3, 4, 5], <function <lambda> at 0x0000029B0D822E80>) = [2, 4]


## Exercise 4 – Generator Expression Pipeline

**Goal**: Build a **lazy pipeline** using only **generator expressions** (no generator functions): start from a list of numbers, filter to keep only positive ones, then map each to its square. Consume the pipeline (e.g. with `list(...)`) and print the result.

Requirements:

- Use the form `(expr for x in iterable if condition)` for filtering and a similar form for mapping.
- You may chain two generator expressions (e.g. assign the first to a variable, then feed it into the second), or write a single expression with both filter and map.
- Start from something like `data = [-2, 1, 3, -1, 4]` and show the final sequence (e.g. [1, 9, 16]).

Hint: Generator expressions are lazy; chaining them keeps the pipeline lazy until you consume with `list()` or a loop.

In [21]:
# Exercise 4 – generator expression pipeline

data = [-2, 1, 3, -1, 4]

# Demo: print(list(...))  ->  [1, 9, 16]
generator = (x ** 2 for x in data if x >= 0)
print(f"gen({data}) = {list(generator)}")

gen([-2, 1, 3, -1, 4]) = [1, 9, 16]


## Exercise 5 – Read Lines from a File-Like Object

**Goal**: Write a generator function that takes a **file-like object** (e.g. from `open()` or `io.StringIO`) and yields **non-empty, stripped lines**, skipping lines that are empty or that start with `#`.

Requirements:

- Signature: `read_lines(f)` where `f` has a `.readline()` or is iterable line-by-line.
- Yield each valid line as a string (stripped of leading/trailing whitespace).
- Do not read the entire file into memory; iterate line by line.

Hint: `for line in f:` then strip and skip blanks/comments.

In [28]:
# Exercise 5 – implement read_lines(f) here

import io
from collections.abc import Iterator

def read_lines(stream: io.StringIO) -> Iterator[str]:
    for line in stream:
        if line.startswith("#"): continue
        line = line.strip(" \n")
        if line: yield line

sample = """# header
first

second
# comment
third
"""

# Demo: list(read_lines(io.StringIO(sample)))  ->  ['first', 'second', 'third']

result = read_lines(arg := io.StringIO(sample))
print(f"read_lines(...) = {list(result)}")

read_lines(...) = ['first', 'second', 'third']


## Data sampler

This section was written by the human user of this notebook. Its purpose is to create a price history based on a predefined set of seed values for a given array of real FX symbols and a random walk generator. The resulting dataframe `df_sample` can be afterwards converted to a string, and used as a `StringIO` sample to iterate on.

In [151]:
import numpy, pandas

max_spread = 4
ms = pandas.Timedelta(milliseconds = 1)
seed_ts = pandas.Timestamp.utcnow().round(ms)
seed = {"AUDUSD": 0.65171, "EURUSD": 1.16397, "GBPUSD": 1.34529, "NZDUSD": 0.59522,
        "USDCAD": 1.37582, "USDCHF": 0.80774, "USDJPY": 147.656, "XAUUSD": 5018.66}

df_sample = dict.fromkeys(seed)
for symbol, value in seed.items():
    array = numpy.random.randint(1, 1000, 10000)
    df = pandas.DataFrame({"ts": array * ms})
    df.index = df.pop("ts").cumsum() + seed_ts
    array = numpy.random.random(df.shape[0])
    delta = pow(10, round(numpy.log10(value) - 5))
    df["bid"] = numpy.sign(array * 2 - 1) * delta
    df["bid"] = df["bid"].cumsum() + value
    array = numpy.random.random(df.shape[0])
    df["ask"] = numpy.round(array * max_spread + 1)
    df["ask"] = df["ask"] * delta + df["bid"]
    df_sample[symbol] = df
    
df_sample: pandas.DataFrame = pandas.concat(df_sample)
df_sample = df_sample.rename_axis(["symbol", "ts"])

MASTER_SEP, MASTER_PRINT_END = ",", 6 * "\t"
MASTER_TS_FORMAT = "%Y-%m-%dT%H:%M:%S.%f"

def generate_csv(both_prices: bool = True) -> str:

    df = df_sample.reset_index()
    df["ts"] = df["ts"].dt.strftime(MASTER_TS_FORMAT)
    df["bid"] = df["bid"].astype(str).str[: 7]
    df["ask"] = df["ask"].astype(str).str[: 7]
    if not both_prices: df.pop("bid")
    df["str"] = df.apply(MASTER_SEP.join, axis = "columns")
    return str.join("\n", df["str"])

## Exercise 6 – Parse CSV-Like Lines into Objects

**Goal**: Define a simple **dataclass** (e.g. `Tick` with fields `symbol`, `price`, `timestamp` or similar) and a generator that reads from a file-like object containing **CSV-like lines** (e.g. `symbol,price,ts`) and **yields** parsed `Tick` instances line by line.

Requirements:

- Reuse a “read lines” idea (skip blanks/comments) then split each line by comma.
- Parse the fields (e.g. symbol as str, price as float, timestamp as str or datetime) and yield a `Tick` (or named tuple) per line.
- Do not load all lines into a list first; yield as you go.

Hint: You can combine “read lines” and “split + parse” in one generator, or call a helper generator for lines.

In [160]:
# Exercise 6 – Tick dataclass and iter_ticks(f) or similar

import io
from dataclasses import dataclass
from datetime import datetime

@dataclass
class Tick:
    symbol: str
    timestamp: datetime
    price: float

    TS_FORMAT = MASTER_TS_FORMAT

    def __repr__(self) -> str:
        ts_str = self.timestamp.isoformat(" ", "milliseconds")
        return f"Tick({ts_str}, {self.symbol}, {self.price})"

def iter_ticks(stream: io.StringIO) -> Iterator[Tick]:
    for line in stream:
        if line.startswith("#"): continue
        symbol, ts_str, price = line.strip().split(",")
        ts = datetime.strptime(ts_str, Tick.TS_FORMAT)
        yield Tick(symbol, ts, float(price))


csv_sample = generate_csv(both_prices = False)

# Demo: for t in iter_ticks(io.StringIO(csv_sample)): print(t)
result = iter_ticks(io.StringIO(csv_sample))
for n_tick, tick in enumerate(result, 1):
    print(f"\rTick #{n_tick}: {tick}", end = MASTER_PRINT_END)

Tick #80000: Tick(2026-03-14 09:31:49.671, XAUUSD, 5007.86)																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																																												

## Exercise 7 – Chained Generators (Filter + Map)

**Goal**: Build a small **pipeline** of generator functions: (1) a source that yields simple tick-like objects (e.g. symbol, bid, ask), (2) a **filter** generator that yields only ticks for a given symbol, (3) a **map** generator that converts each tick to a mid-price (e.g. `(bid+ask)/2`). Compose them and iterate over the result.

Requirements:

- Use in-memory data (a list of ticks) as the source; wrap it in a generator that yields one tick at a time.
- Filter and map should be separate generator functions that take an iterable and yield items.
- Pipeline: source → filter(symbol) → to_midprice; then loop and print (or collect) the mid-prices.

Hint: Each stage is a generator that loops over its input iterable and yields transformed/filtered items.

In [159]:
# Exercise 7 – iter_ticks, filter_symbol, to_midprice; compose and run

from typing import TextIO, Iterator
from dataclasses import dataclass
from datetime import datetime
import time

@dataclass
class Tick:
    timestamp: datetime
    symbol: str
    ask: float
    bid: float

    TS_FORMAT = MASTER_TS_FORMAT

    def __repr__(self) -> str:
        prices = f"A{self.ask:.5f} B{self.bid:.5f}"
        ts_str = self.timestamp.isoformat(" ", "milliseconds")
        return f"Tick({ts_str} @ {self.symbol}, {prices})"

    @property
    def midprice(self):
        midprice = (self.ask + self.bid) / 2
        return round(midprice, 6)

def iter_ticks(stream: TextIO) -> Iterator[TextIO]:
    for line in stream:
        if line.startswith("#"): continue
        symbol, ts_str, ask, bid = line.strip().split(",")
        ts = datetime.strptime(ts_str, Tick.TS_FORMAT)
        yield Tick(ts, symbol, float(ask), float(bid))

def filter_symbol(stream: Iterator[Tick], symbols: set) -> Iterator[Tick]:
    if isinstance(symbols, str): symbols = {symbols}
    for tick in stream:
        if tick.symbol not in symbols: continue
        yield tick

def to_midprice(stream: Iterator[Tick]) -> Iterator[float]:
    for tick in stream: yield tick.midprice


csv_sample = generate_csv(both_prices = True)

# Small in-memory tick list, then: pipeline = to_midprice(filter_symbol(iter_ticks(ticks), "AAPL"))
symbols = {"AUDUSD", "EURUSD"}
pipeline_iter_ticks = iter_ticks(io.StringIO(csv_sample))
pipeline_filter_symbol = filter_symbol(pipeline_iter_ticks, symbols)
pipeline_to_midprice = to_midprice(pipeline_filter_symbol)
for n, mp in enumerate(pipeline_to_midprice, 1):
    print(f"\rTick #{n}, midprice = {mp}", end = "\t" * 6)
    time.sleep(1e-4)

Tick #20000, midprice = 1.16294							

## Exercise 8 – Sliding Window Generator

**Goal**: Implement a generator that takes an iterable and a **window size** and yields **successive overlapping windows** as tuples. For example, `window([1,2,3,4,5], 3)` yields `(1,2,3)`, then `(2,3,4)`, then `(3,4,5)`.

Requirements:

- Signature: `window(iterable, size: int)`.
- Each yielded value is a tuple of length `size`.
- Use only O(size) extra memory (e.g. a `collections.deque` with `maxlen=size`).
- If the iterable has fewer than `size` elements, you may yield nothing or yield a shorter tuple; specify your choice in a one-line comment.

Hint: `deque(iterable, maxlen=size)` does not slide; you need to append one item at a time and yield `tuple(deque)` when the deque is full.

In [129]:
# Exercise 8 – implement window(iterable, size) here

from collections import deque
from typing import Iterator

def generator(array: list, size: int) -> Iterator[tuple[int]]:
    queue = deque(list(), maxlen = size)
    for element in array:
        queue.append(element)
        if (len(queue) >= queue.maxlen):
            yield tuple(queue)

# Demo: list(window([1,2,3,4,5], 3))  ->  [(1,2,3), (2,3,4), (3,4,5)]
array, size = [1,2,3,4,5], 3
result = generator(array, size)
print(f"generator({array}, {size}) = {list(result)}")

generator([1, 2, 3, 4, 5], 3) = [(1, 2, 3), (2, 3, 4), (3, 4, 5)]


## Exercise 9 – Flatten Multiple Iterables with `yield from`

**Goal**: Write a generator function that takes an **iterable of iterables** (e.g. a list of lists, or a list of generators) and yields all elements from the first iterable, then all from the second, and so on—i.e. **flatten** one level.

Requirements:

- Signature: `flatten(iterables)`.
- Use `yield from` to delegate to each inner iterable.
- Do not build one big list; yield one item at a time.

Hint: `for it in iterables: yield from it`.

In [131]:
# Exercise 9 – implement flatten(iterables) here

from typing import Iterator

def flatten(iterables: Iterator[Iterator]) -> Iterator[list]:
    for it in iterables: yield from it

# Demo: list(flatten([[1,2], [3], [4,5]]))  ->  [1, 2, 3, 4, 5]
iterables = [[1,2], [3], [4,5]]
result = flatten(iterables)
print(f"flatten({iterables}) = {list(result)}")

flatten([[1, 2], [3], [4, 5]]) = [1, 2, 3, 4, 5]
